In [27]:
import numdifftools as nd
import pandas as pd
import numpy as np
from scipy.optimize import minimize_scalar

# Идея методов штрафных и барьерных функций в оптимизации

Эти методы используются для решения задач **оптимизации с ограничениями**, преобразуя их в последовательность задач **безусловной оптимизации**. Их основная цель — учесть ограничения через добавление специальных слагаемых к целевой функции.

# Метод барьерных поверхностей

In [28]:
def fast_descent_method(f, x0, check_restrictions):
    grad = nd.Gradient(f)(x0)
    l, r = 0, 1e9
    
    while r - l > 0.001:
        mid = (r + l) / 2.
        if check_restrictions(x0 - mid * grad):
            l = mid
        else:
            r = mid

    left_border, right_border = 0, l
    if left_border > right_border:
        left_border, right_border = right_border, left_border

    val = minimize_scalar(lambda l: f(x0 - l * grad), bounds=[left_border, right_border], method='bounded').x
    return x0 - val * grad

**Идея:** Добавляют слагаемое, которое стремится к бесконечности при приближении к границе допустимой области, "отталкивая" решение от нарушений.  

**Типы барьеров:**  
1. **Логарифмический барьер** для $ g(x) \leq 0$:  
   $
   B(x) = f(x) - \mu \cdot \sum \ln(-g_i(x)),
   $  
   где $\mu \to 0$.  
   - **Особенности:**  
     - Плавный рост штрафа при приближении к границе.  
     - Гарантирует гладкость функции.  
     - Широко используется в методах внутренней точки (interior-point).  

2. **Обратный барьер** для $ g(x) \leq 0$:  
   $
   B(x) = f(x) + \mu \cdot \sum \frac{1}{-g_i(x)},
   $  
   где $\mu \to 0$.  
   - **Особенности:**  
     - Резкий рост штрафа около границы (быстрее, чем логарифмический).  
     - Менее гладкий, чем логарифмический барьер.  
     - Может вызывать численные проблемы из-за больших градиентов.  

**Сравнение барьеров:**  
| **Критерий**      | **Логарифмический барьер**         | **Обратный барьер**             |  
|--------------------|------------------------------------|----------------------------------|  
| **Рост штрафа**    | Плавный ($-\ln(-g(x))$)         | Резкий ($\frac{1}{-g(x)}$)    |  
| **Гладкость**      | Гладкая функция                   | Негладкая при $g(x) \to 0$     |  
| **Устойчивость**   | Лучше для методов Ньютона         | Риск численной неустойчивости    |  
| **Применение**     | Современные задачи (IPOPT, NLP)   | Специфические задачи с жёсткими ограничениями |  

**Преимущества барьерных методов:**  
- Гарантируют нахождение решения внутри допустимой области.  
- Эффективны с методами внутренней точки.  

**Недостатки:**  
- Требуют начальную точку внутри допустимого множества.  
- Численные проблемы при малых $\mu$ (особенно для обратного барьера).  


In [29]:
def barrier_functions(f, g, x, penalty_type, r=10, c=12, eps=0.01):
    '''
    f - исследуемая функция
    g - список ограничений-неравенств вида g_i <= 0
    x - начальная точка
    penalty_type - вид штрафной функции
    r (>= 0) - начальное значение параметра штрафа (обычно 1, 10, 100)
    c - число для уменьшения параметра штрафа (обычно 10, 12, 16)
    eps - малое число для остановки алгоритма
    '''
    # для сбора данных
    data = {
        'r': [],
        'x_n': [],
        'y_n': [],
        'f(x_n, y_n)': [],
        'F': []
    }
    
    # проверка точки на допустимость
    point_in_set = lambda x: all(func(x) <= 0 for func in g)

    # Составление штрафной функции
    penalty_funcs = {
        'inverse' : lambda f: lambda x: 1/f(x),
        'logarithmic' : lambda f: lambda x: np.log(-f(x))
    }

    penalty_f = lambda r: lambda x: -r * sum(penalty_funcs[penalty_type](f)(x) for f in g)

    while abs(penalty_f(r)(x)) > eps:
        # Составление вспомогательной функции F
        F = lambda x: f(x) + penalty_f(r)(x)
        # Нахождение точки безусловного минимума функции F
        # с проверкой на принадлеждость текущей точки внутренности множества X
        x = fast_descent_method(F, x, point_in_set)
        # Уменьшение значения штрафа
        r /= c

        # Добавление данных
        data['r'].append(r)
        data['x_n'].append(x[0])
        data['y_n'].append(x[1])
        data['f(x_n, y_n)'].append(f(x))
        data['F'].append(F(x))

    df = pd.DataFrame(data)
    print(df)

    return x

# Метод штрафных функций

In [30]:
def Gauss_Zeidel(func: callable, x: list, eps: float) -> float:
    results = []
    n = len(x)
    j = 0

    while True:
        for i in range(n):
            e = np.zeros(n)
            e[i] = 1

            _lambda = minimize_scalar(lambda _lambda: func(x + _lambda * e)).x
            x[i] += _lambda

        results.append(func(x))

        if j >= 1 and abs(results[j] - results[j-1]) < eps:
            return x
            
        j += 1

**Идея:** Нарушения ограничений "штрафуются" добавлением к целевой функции слагаемого, которое растёт при отклонении от допустимой области.  
**Типы штрафов:**  
- **Внешние штрафы:** Работают вне допустимой области. Пример: квадратичный штраф для ограничения $ g(x) \leq 0 $:  
  $
  P(x) = f(x) + \mu \cdot \max(0, g(x))^2,
  $  
  где $\mu$ — параметр штрафа (увеличивается на каждой итерации).  
- **Внутренние штрафы:** Реже используются (чаще относят к барьерным методам).
 
**Преимущества:**  
- Не требуют начальной точки внутри допустимой области.  
- Просты в реализации.  

**Недостатки:**  
- Медленная сходимость при больших $\mu$.  
- Плохая обусловленность задачи.

In [31]:
def penalty_functions(f, h, g, x, r=0.01, c=5, eps=0.01):
    '''
    f - исследуемая функция
    h - список ограничений-равенств
    g - список ограничений-неравенств вида g_i <= 0
    x - начальная точка
    r (> 0) - начальное значение параметра штрафа (обычно маленькое)
    c ∈ [4, 10] - число для увеличения параметра
    eps - малое число для остановки алгоритма
    '''
    # для сбора данных
    data = {
        'r': [],
        'x_n': [],
        'y_n': [],
        'f(x_n, y_n)': [],
        'F': []
    }

    # Составление штрафной функции
    penalty_h = lambda x: sum(f(x)**2 for f in h) # или модуль можно
    penalty_g = lambda x: sum(max(0, f(x))**2 for f in g) # срезка ^ 2

    penalty_f = lambda r: lambda x: r/2 * (penalty_h(x) + penalty_g(x))

    # Добавляем начальные значения
    data['r'].append(r)
    data['x_n'].append(x[0])
    data['y_n'].append(x[1])
    data['f(x_n, y_n)'].append(f(x))
    data['F'].append(np.inf)

    while penalty_f(r)(x) > eps:
        # Составление вспомогательной функции F
        F = lambda x: f(x) + penalty_f(r)(x)
        # Нахождение точки безусловного минимума функции F
        x = Gauss_Zeidel(F, x, eps)
        # Увеличение значения штрафа
        r *= c

        # Добавляем данные текущей итерации
        data['r'].append(r)
        data['x_n'].append(x[0])
        data['y_n'].append(x[1])
        data['f(x_n, y_n)'].append(f(x))
        data['F'].append(F(x))

    # Создаем DataFrame и выводим его
    df = pd.DataFrame(data)
    print(df)

    return x

# Проверка алгоритмов

In [32]:
f = lambda x: x[0] + x[1]

g1 = lambda x: x[0]**2 - x[1]
g2 = lambda x: -x[0]
g = [g1, g2]

x0 = [1, 4]
eps = 1e-3
r = 1
c = 12

print('С использованием обратной штрафной функции: ')
res1 = barrier_functions(f, g, x0, 'inverse', r, c, eps)
print(f"\nМинимальное значение в точке: ({float(res1[0]):3f}, {float(res1[1]):3f})\n\n")

print('С использованием логарифмической штрафной функции: ')
res2 = barrier_functions(f, g, x0, 'logarithmic', r, c, eps)
print(f"\nМинимальное значение в точке: ({float(res2[0]):3f}, {float(res2[1]):3f})")

С использованием обратной штрафной функции: 
          r       x_n       y_n  f(x_n, y_n)         F
0  0.083333  0.498983  1.995932     2.494915  2.709624
1  0.006944  0.187612  1.558613     1.746225  1.787798
2  0.000579  0.055725  1.395029     1.450754  1.461555
3  0.000048  0.016116  1.346365     1.362480  1.365509
4  0.000004  0.004653  1.332288     1.336940  1.337807

Минимальное значение в точке: (0.004653, 1.332288)


С использованием логарифмической штрафной функции: 
          r       x_n       y_n  f(x_n, y_n)         F
0  0.083333  0.500000  3.500000     4.000000  3.959541
1  0.006944  0.039592  2.977746     3.017339  3.032189
2  0.000579  0.003147  2.933662     2.936810  2.939521
3  0.000048  0.000259  2.930124     2.930383  2.930729

Минимальное значение в точке: (0.000259, 2.930124)


In [33]:
f = lambda x: x[0]**2 + x[1]**2

h1 = lambda x: x[0] - 1
h = np.array([h1])

g1 = lambda x: x[0] + x[1] - 2
g = np.array([g1])

x0 = [2, 2]
eps = 0.01
r = 0.1
c = 10

res1 = penalty_functions(f, h, g, x0, r, c, eps)
print(f"\n\nМинимальное значение в точке: ({float(res1[0]):3f}, {float(res1[1]):3f})")

          r       x_n           y_n  f(x_n, y_n)         F
0       0.1  2.000000  2.000000e+00     8.000000       inf
1       1.0  0.047619  3.195989e-11     0.002268  0.455782
2      10.0  0.333333 -4.374232e-09     0.111111  2.333333
3     100.0  0.833333 -4.338739e-09     0.694444  2.083333
4    1000.0  0.980392 -4.303534e-09     0.961169  1.153403
5   10000.0  0.998004 -4.268615e-09     0.996012  1.015932
6  100000.0  0.999800 -4.233979e-09     0.999600  1.001599


Минимальное значение в точке: (0.999800, -0.000000)


#### **Сравнение штрафных и барьерных методов**  
| **Критерий**          | **Штрафные методы**                | **Барьерные методы**               |  
|------------------------|-------------------------------------|-------------------------------------|  
| **Область работы**     | Внешняя или внутренняя             | Только внутри допустимой области   |  
| **Начальная точка**    | Любая                             | Должна быть внутри                 |  
| **Тип ограничений**    | Нежёсткие (допустимы нарушения)   | Жёсткие (нарушения запрещены)      |  
| **Сходимость**         | Медленнее, зависит от $\mu$     | Быстрее при правильном $\mu$     |  
| **Численная устойчивость** | Проблемы при больших $\mu$    | Проблемы при малых $\mu$         |  